## Verificar Roteamento de Proxy do Browser

Este notebook verifica se o AgentCore Browser roteia o tráfego através do proxy Squid
implantado pelo `agentcore-browser-proxy.yaml`.

A stack implanta:
- **Proxy Squid** no EC2 em uma subnet pública (com autenticação básica via Secrets Manager)
- **AgentCore Browser** em uma subnet privada (modo VPC, saída bloqueada para Squid:3128)
- **Bucket S3** recebendo logs de acesso do Squid a cada 5 minutos

### Pré-requisitos

1. Implantar a stack CloudFormation `agentcore-browser-proxy.yaml`
2. Instalar dependências e **reiniciar seu kernel**:

In [ ]:
!pip install -qU -r requirements.txt

### 1. Ler saídas do CloudFormation

Obter o ID do Browser, IPs do Squid e ARN do Secrets Manager da stack.

In [ ]:
import boto3
from urllib.parse import urlparse
from botocore.auth import SigV4Auth
from botocore.awsrequest import AWSRequest

STACK_NAME = "agentcore-browser-proxy"

session = boto3.Session()
REGION = session.region_name
print(f"Região: {REGION}")
browser_client = session.client("bedrock-agentcore")

cfn = session.client("cloudformation")
outputs = {o["OutputKey"]: o["OutputValue"]
           for o in cfn.describe_stacks(StackName=STACK_NAME)["Stacks"][0]["Outputs"]}

BROWSER_ID = outputs["BrowserId"]
SQUID_IP = outputs["SquidPrivateIp"]
SQUID_PUBLIC_IP = outputs["SquidPublicIp"]
SECRET_ARN = outputs["ProxySecretArn"]
LOG_BUCKET = outputs["LogBucketName"]

print(f"Browser ID:       {BROWSER_ID}")
print(f"IP privado Squid: {SQUID_IP}")
print(f"IP público Squid: {SQUID_PUBLIC_IP}")
print(f"Bucket de logs:   {LOG_BUCKET}")

### 2. Iniciar uma sessão de navegador com proxy

Construir a `proxyConfiguration` apontando para a instância Squid e iniciar uma sessão.
O navegador irá rotear todo o tráfego web através do proxy.

In [ ]:
proxy_config = {
    "proxies": [{
        "externalProxy": {
            "server": SQUID_IP,
            "port": 3128,
            "credentials": {
                "basicAuth": {"secretArn": SECRET_ARN}
            },
        }
    }]
}

response = browser_client.start_browser_session(
    browserIdentifier=BROWSER_ID,
    proxyConfiguration=proxy_config,
)
session_id = response["sessionId"]
ws_url = (
    f"wss://bedrock-agentcore.{REGION}.amazonaws.com"
    f"/browser-streams/{BROWSER_ID}/sessions/{session_id}/automation"
)
print(f"Session ID: {session_id}")

# Assinar a URL WebSocket com SigV4
credentials = session.get_credentials()
https_url = ws_url.replace("wss://", "https://")
parsed = urlparse(https_url)
request = AWSRequest(method="GET", url=https_url, headers={"host": parsed.netloc})
SigV4Auth(credentials, "bedrock-agentcore", REGION).add_auth(request)
headers = {k: v for k, v in request.headers.items()}

### 3. Verificar roteamento do proxy

Conectar via Playwright e navegar para um serviço de detecção de IP.
Se o proxy estiver funcionando, o IP observado deve corresponder ao IP público da instância Squid.

In [ ]:
from playwright.async_api import async_playwright

async with async_playwright() as p:
    browser = await p.chromium.connect_over_cdp(ws_url, headers=headers)
    page = (
        browser.contexts[0].pages[0]
        if browser.contexts
        else await browser.new_context().new_page()
    )

    print("Verificando IP público do navegador via icanhazip.com...")
    await page.goto("https://icanhazip.com", timeout=15000, wait_until="domcontentloaded")
    observed_ip = (await page.inner_text("body")).strip()

    print(f"\n{'=' * 50}")
    print(f"IP Esperado (público Squid): {SQUID_PUBLIC_IP}")
    print(f"IP Observado (navegador):    {observed_ip}")
    match = observed_ip == SQUID_PUBLIC_IP
    print(f"Resultado: {'PASS' if match else 'FAIL'} — tráfego {'está' if match else 'NÃO está'} roteado através do proxy")
    print(f"{'=' * 50}")

    await browser.close()

### 4. Encerrar a sessão

In [ ]:
browser_client.stop_browser_session(browserIdentifier=BROWSER_ID, sessionId=session_id)
print(f"Sessão {session_id} encerrada")

### Solução de Problemas

- **Incompatibilidade de IP**: Verifique se o grupo de segurança do navegador só permite saída para Squid:3128
- **Timeout de conexão**: Verifique se o Squid está rodando — SSH para a instância EC2 e verifique `systemctl status squid`
- **Erros de autenticação**: Verifique se o secret do Secrets Manager corresponde ao htpasswd do Squid — verifique `/var/log/squid/access.log` na instância
- **Sem logs no S3**: Os logs sincronizam a cada 5 minutos via cron — verifique `/var/log/user-data.log` na instância para erros de configuração